# Análise - Rating Books

Respondendo às perguntas:
1. Quais gêneros têm mais avaliações no total e melhores notas médias?
2. Quais livros são os mais bem avaliados dentro de cada gênero?
3. Quais autores se destacam (melhor nota média) dentro de cada gênero?
4. Quais os livros são mais caros?

In [ ]:
import pandas as pd
import ast
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("books_1.Best_Books_Ever.csv")
df["genres"] = df["genres"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df_exploded = df.explode("genres")

## 1. Gêneros com mais avaliações

In [ ]:
genre_stats = df_exploded.groupby("genres").agg(
    total_ratings=("numRatings", "sum"),
    avg_rating=("rating", "mean"),
    book_count=("title", "nunique"),
).sort_values("total_ratings", ascending=False)

display("Gêneros com mais avaliações (TOP 15):")
display(genre_stats[["total_ratings", "avg_rating", "book_count"]].head(15))

In [ ]:
top15 = genre_stats.head(15).sort_values("total_ratings").reset_index()

fig = px.bar(
    top15, x="total_ratings", y="genres",
    orientation="h", text_auto=".2s",
    title="Gêneros com Mais Avaliações",
    color_discrete_sequence=["#4E79A7"],
    labels={"total_ratings": "Total de Avaliações", "genres": ""}
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#555555"),
    title=dict(font=dict(size=18, color="#333333")),
    xaxis=dict(gridcolor="#eeeeee", showgrid=True),
    yaxis=dict(autorange="reversed"),
    height=500,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig.show()

## 1b. Gêneros com melhores notas médias

In [ ]:
reliable = genre_stats[genre_stats["book_count"] >= 5].sort_values("avg_rating", ascending=False).head(15)

display("Gêneros com melhores notas médias (mín. 5 livros, TOP 15):")
display(reliable[["avg_rating", "total_ratings", "book_count"]])

In [ ]:
top_avg = reliable.sort_values("avg_rating").reset_index()

fig = px.bar(
    top_avg, x="avg_rating", y="genres",
    orientation="h", text_auto=".2f",
    title="Gêneros com Melhores Notas (mín. 5 livros)",
    color="avg_rating",
    color_continuous_scale=["#4E79A7", "#E15759"],
    range_color=[4.0, 4.7],
    labels={"avg_rating": "Nota Média", "genres": ""}
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#555555"),
    title=dict(font=dict(size=18, color="#333333")),
    xaxis=dict(gridcolor="#eeeeee", showgrid=True, range=[4.0, 4.7]),
    yaxis=dict(autorange="reversed"),
    height=500,
    margin=dict(l=20, r=20, t=50, b=20),
    coloraxis_showscale=False
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

## 2. Livros mais bem avaliados dentro de cada gênero

In [ ]:
df_exploded["rank_in_genre"] = df_exploded.groupby("genres")["rating"].rank("dense", ascending=False)
top3 = df_exploded[df_exploded["rank_in_genre"] <= 3][["genres", "title", "author", "rating"]].drop_duplicates()
display("TOP 3 livros por gênero (amostra):")
display(top3.groupby("genres").head(3).head(30))

## 3. Autores que se destacam por gênero

In [ ]:
author_genre = df_exploded.groupby(["genres", "author"]).agg(
    avg_rating=("rating", "mean"),
    book_count=("title", "nunique"),
).reset_index()
author_genre = author_genre[author_genre["book_count"] >= 2]
top_authors = author_genre.loc[author_genre.groupby("genres")["avg_rating"].idxmax()]
top_authors = top_authors.sort_values("avg_rating", ascending=False).head(20)

display("Autores com melhor nota média por gênero (mín. 2 livros, TOP 20):")
display(top_authors[["genres", "author", "avg_rating", "book_count"]])

In [ ]:
top_authors_sorted = top_authors.sort_values("avg_rating").reset_index(drop=True).copy()
import numpy as np
mask = top_authors_sorted["author"].str.len() > 35
top_authors_sorted["label"] = np.where(mask, top_authors_sorted["author"].str[:35] + "...", top_authors_sorted["author"].str[:35])

fig = px.bar(
    top_authors_sorted, x="avg_rating", y="label",
    orientation="h", text_auto=".2f",
    title="Melhores Autores por Gênero",
    color="avg_rating",
    color_continuous_scale="blues",
    labels={"avg_rating": "Nota Média", "label": "Autor"}
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#555555"),
    title=dict(font=dict(size=18, color="#333333")),
    xaxis=dict(gridcolor="#eeeeee", showgrid=True),
    yaxis=dict(autorange="reversed"),
    height=600,
    margin=dict(l=20, r=20, t=50, b=20),
    coloraxis_showscale=False
)
fig.update_traces(textposition="outside")
fig.show()

## 4. Livros mais caros

In [ ]:
expensive = df.dropna(subset=["price"]).nlargest(20, "price")
display("Livros mais caros (TOP 20):")
display(expensive[["title", "author", "price", "rating", "numRatings"]])

In [ ]:
top_expensive = expensive.sort_values("price").reset_index(drop=True).copy()
import numpy as np
mask_price = top_expensive["title"].str.len() > 50
top_expensive["short_title"] = np.where(mask_price, top_expensive["title"].str[:50] + "...", top_expensive["title"].str[:50])

fig = px.bar(
    top_expensive, x="price", y="short_title",
    orientation="h", text_auto=".0f",
    title="Livros Mais Caros",
    color_discrete_sequence=["#76B7B2"],
    labels={"price": "Preço ($)", "short_title": ""}
)
fig.update_traces(
    texttemplate="$%{x:.0f}", textposition="outside"
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#555555"),
    title=dict(font=dict(size=18, color="#333333")),
    xaxis=dict(gridcolor="#eeeeee", showgrid=True),
    yaxis=dict(autorange="reversed"),
    height=600,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig.show()

## Extra: Preço vs Nota vs Avaliações

In [ ]:
sample = df[df["numRatings"] > 1000].sample(min(3000, len(df)), random_state=42).copy()
sample["has_price"] = sample["price"].notna()

fig = px.scatter(
    sample, x="numRatings", y="rating",
    color="price", color_continuous_scale="viridis",
    title="Relação: Avaliações vs Nota (cor = preço)",
    labels={
        "numRatings": "Número de Avaliações",
        "rating": "Nota Média",
        "price": "Preço ($)"
    },
    opacity=0.5,
    size_max=5
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#555555"),
    title=dict(font=dict(size=18, color="#333333")),
    xaxis=dict(gridcolor="#eeeeee", showgrid=True, type="log"),
    yaxis=dict(gridcolor="#eeeeee", showgrid=True),
    height=600,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig.show()

In [ ]:
# Percentual de livros com preço disponível
livros_com_preco = df["price"].notna().sum()
total_livros = len(df)
print(f"Livros com preço: {livros_com_preco} de {total_livros} ({100*livros_com_preco/total_livros:.1f}%)")